In [43]:
import os 
from typing import TypedDict,List
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage,SystemMessage


# langgraph
from langgraph.graph import START,END,StateGraph


# for Rag
from langchain_community.document_loaders import WebBaseLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# vector
from langchain_community.vectorstores import FAISS


from dotenv import load_dotenv

load_dotenv()
os.environ[("OPENAI_API_KEY")]=os.getenv("OPENAI_API_KEY")

model=init_chat_model("gpt-4o")

In [58]:
# load the documents from the Documents folder 
documents=[]
folder_path="./Documents"

def load_folder_pdf(folder_path:str):
    all_doc=[]
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path=os.path.join(folder_path,filename)
            print(f"Loading: {filename}")
            
            loader=PyPDFLoader(file_path)
            docs=loader.load()
            all_doc.extend(docs)
    
    return all_doc

documents=load_folder_pdf(folder_path)
print(f"Loaded {len(documents)} documents")

Loading: Code_Agent_Final_Roadmap.pdf.pdf
Loading: DS -Intern Assignment .pdf
Loaded 37 documents


In [39]:
# split the doc into chunks
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks=text_splitter.split_documents(documents)

In [ ]:
# init the embedding modle 
embedding=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=FAISS.from_documents(chunks,embedding)

# retriver
retriver=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
class App_State(TypedDict):
    user_question:str
    relivent_chunks:List[Document]
    answer:str

In [56]:
# retrive the relevant chunks based on the user question
def retriveer_node(state:App_State):
    user_question=state["user_question"]
    print("retriver fxn has been caleed ")
    retrived_chunks=retriver.invoke(user_question)
    return {"relivent_chunks":retrived_chunks}

def generate_node(state:App_State):
    print("generate node fxn has been caleed ")
    user_question=state["user_question"]
    context=""
    for chunk in state["relivent_chunks"]:
        context+=chunk.page_content+"\n\n"

    sys_msg=SystemMessage(content="You are a helpful assistant. only answer the quesion from the given context. if you don't find the answer there. just say i don't know")
    human_msg=HumanMessage(content=f"Answer this user question->{user_question} \n\n based on the context:\n\n{context}")
    
    model_res=model.invoke([sys_msg,human_msg]).content
    return{"answer":model_res}

In [57]:
# build the graph
graph=StateGraph(App_State)
graph.add_node("Retrive",retriveer_node)
graph.add_node("Generate",generate_node)

graph.add_edge(START,"Retrive")
graph.add_edge("Retrive","Generate")
graph.add_edge("Generate",END)

# compile the graph
workflow=graph.compile()

In [58]:
init_state={"user_question":"what is my assinment task ?"}
res=workflow.invoke(init_state)

retriver fxn has been caleed 
generate node fxn has been caleed 


In [ ]:
res["answer"]

'Your assignment task is to build a Retrieval-Augmented Generation (RAG) system with a self-corrective LangGraph workflow. The system should act as a technical documentation assistant that answers questions about a set of technical documents. This involves using a LangGraph workflow with retrieval, document grading, and answer generation nodes, and serving the system via a FastAPI application. Specific API endpoints you need to implement include /query, /ingest, /documents, and /feedback, among others.'